# ⏱️ Rate Limiting

**Protect your API from abuse and overuse**

---

## 📋 Overview

**What you'll learn:**
- Rate limiting strategies
- Token bucket algorithm
- Sliding window
- Redis-based rate limiting
- Per-user and per-endpoint limits

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
# Installation
# !pip install fastapi uvicorn slowapi redis

from fastapi import FastAPI, HTTPException, Request, Depends
from typing import Dict, Optional
import time
from datetime import datetime, timedelta
from collections import defaultdict
import asyncio

print("✅ Setup complete")

## 🤔 Why Rate Limiting?

### Without Rate Limiting:

```
❌ Problems:
- API abuse (spam, bots)
- Cost explosion (OpenAI bills)
- Server overload
- Poor user experience for everyone
- DDoS attacks
- No fair usage
```

### With Rate Limiting:

```
✅ Benefits:
- Prevent abuse
- Control costs
- Fair resource allocation
- Better performance
- Predictable behavior
- Tiered pricing support
```

### Common Limits:

| Service | Free Tier | Paid Tier |
|---------|-----------|------------|
| **OpenAI** | 3 RPM | 3,500+ RPM |
| **GitHub** | 60/hour | 5,000/hour |
| **Twitter** | 15/15min | 900/15min |
| **Your API** | ? | ? |

## 🪣 Token Bucket Algorithm

In [ ]:
import time
from typing import Dict

class TokenBucket:
    """Token bucket rate limiter."""
    
    def __init__(self, capacity: int, refill_rate: float):
        """
        Args:
            capacity: Maximum tokens in bucket
            refill_rate: Tokens added per second
        """
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.tokens = capacity
        self.last_refill = time.time()
    
    def _refill(self):
        """Refill tokens based on time elapsed."""
        now = time.time()
        time_elapsed = now - self.last_refill
        
        # Add tokens based on time elapsed
        tokens_to_add = time_elapsed * self.refill_rate
        self.tokens = min(self.capacity, self.tokens + tokens_to_add)
        self.last_refill = now
    
    def consume(self, tokens: int = 1) -> bool:
        """Try to consume tokens. Returns True if successful."""
        self._refill()
        
        if self.tokens >= tokens:
            self.tokens -= tokens
            return True
        
        return False
    
    def get_wait_time(self, tokens: int = 1) -> float:
        """Get time to wait before tokens are available."""
        self._refill()
        
        if self.tokens >= tokens:
            return 0.0
        
        tokens_needed = tokens - self.tokens
        return tokens_needed / self.refill_rate

# Example usage
print("🪣 Token Bucket Example\n")

# 10 requests per second, burst of 20
bucket = TokenBucket(capacity=20, refill_rate=10.0)

print("Simulating requests:")
for i in range(25):
    if bucket.consume():
        print(f"  Request {i+1}: ✅ Allowed ({bucket.tokens:.1f} tokens left)")
    else:
        wait_time = bucket.get_wait_time()
        print(f"  Request {i+1}: ❌ Rate limited (wait {wait_time:.2f}s)")
        break

print("\n💡 Benefits:")
print("  - Allows bursts")
print("  - Smooth rate limiting")
print("  - Predictable behavior")

## 🪟 Sliding Window Rate Limiter

In [ ]:
from collections import deque
from typing import Deque
import time

class SlidingWindowRateLimiter:
    """Sliding window rate limiter."""
    
    def __init__(self, max_requests: int, window_seconds: int):
        """
        Args:
            max_requests: Max requests per window
            window_seconds: Window size in seconds
        """
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.requests: Deque[float] = deque()
    
    def _clean_old_requests(self):
        """Remove requests outside the window."""
        now = time.time()
        cutoff = now - self.window_seconds
        
        while self.requests and self.requests[0] < cutoff:
            self.requests.popleft()
    
    def is_allowed(self) -> bool:
        """Check if request is allowed."""
        self._clean_old_requests()
        
        if len(self.requests) < self.max_requests:
            self.requests.append(time.time())
            return True
        
        return False
    
    def get_remaining(self) -> int:
        """Get remaining requests in window."""
        self._clean_old_requests()
        return max(0, self.max_requests - len(self.requests))
    
    def get_reset_time(self) -> float:
        """Get time until window resets."""
        if not self.requests:
            return 0.0
        
        oldest_request = self.requests[0]
        reset_time = oldest_request + self.window_seconds
        return max(0, reset_time - time.time())

# Example
print("🪟 Sliding Window Example\n")

# 5 requests per 10 seconds
limiter = SlidingWindowRateLimiter(max_requests=5, window_seconds=10)

print("Making requests:")
for i in range(7):
    if limiter.is_allowed():
        remaining = limiter.get_remaining()
        print(f"  Request {i+1}: ✅ Allowed ({remaining} remaining)")
    else:
        reset_in = limiter.get_reset_time()
        print(f"  Request {i+1}: ❌ Rate limited (reset in {reset_in:.1f}s)")

print("\n💡 Benefits:")
print("  - Precise rate limiting")
print("  - No bursts")
print("  - Fair distribution")

## 🚀 FastAPI with SlowAPI

In [ ]:
print("""
from fastapi import FastAPI, Request
from slowapi import Limiter, _rate_limit_exceeded_handler
from slowapi.util import get_remote_address
from slowapi.errors import RateLimitExceeded

# Create limiter
limiter = Limiter(key_func=get_remote_address)
app = FastAPI()

# Add rate limit exception handler
app.state.limiter = limiter
app.add_exception_handler(RateLimitExceeded, _rate_limit_exceeded_handler)

# Global rate limit
@app.get("/")
@limiter.limit("100/hour")
async def homepage(request: Request):
    return {"message": "Welcome"}

# Different limits per endpoint
@app.post("/api/chat")
@limiter.limit("10/minute")
async def chat(request: Request, message: str):
    return {"response": "..."}

# Expensive endpoint - stricter limit
@app.post("/api/analyze")
@limiter.limit("5/hour")
async def analyze(request: Request, text: str):
    return {"analysis": "..."}

# Multiple rate limits
@app.post("/api/upload")
@limiter.limit("3/minute")
@limiter.limit("100/hour")
@limiter.limit("1000/day")
async def upload(request: Request):
    return {"status": "uploaded"}

# Rate limit formats:
#   - "10/second"  : 10 requests per second
#   - "100/minute" : 100 requests per minute
#   - "1000/hour"  : 1000 requests per hour
#   - "10000/day"  : 10000 requests per day

# Run: uvicorn main:app --reload
""")

## 👤 Per-User Rate Limiting

In [ ]:
print("""
from fastapi import FastAPI, Depends, Request
from slowapi import Limiter
from typing import Dict

app = FastAPI()

# User rate limits by plan
RATE_LIMITS = {
    "free": "10/hour",
    "pro": "100/hour",
    "enterprise": "1000/hour"
}

def get_user_plan(request: Request) -> str:
    \"\"\"Get user plan from API key.\"\"\" 
    api_key = request.headers.get("X-API-Key")
    
    # Look up user plan
    user = get_user_by_api_key(api_key)
    return user.get("plan", "free")

def get_rate_limit_key(request: Request) -> str:
    \"\"\"Create rate limit key from user ID.\"\"\" 
    api_key = request.headers.get("X-API-Key")
    user = get_user_by_api_key(api_key)
    return f"user:{user['user_id']}"

# Create limiter with custom key function
limiter = Limiter(key_func=get_rate_limit_key)
app.state.limiter = limiter

@app.post("/api/chat")
async def chat(request: Request, message: str):
    \"\"\"Chat with per-user rate limit.\"\"\" 
    
    # Get user's rate limit
    plan = get_user_plan(request)
    rate_limit = RATE_LIMITS.get(plan, "10/hour")
    
    # Apply dynamic rate limit
    @limiter.limit(rate_limit)
    async def rate_limited_chat():
        return {"response": "..."}
    
    return await rate_limited_chat()

Example headers:
  X-RateLimit-Limit: 100
  X-RateLimit-Remaining: 95
  X-RateLimit-Reset: 1640000000
""")

## 🔴 Redis-Based Rate Limiting

In [ ]:
print("""
# Production-ready rate limiting with Redis

import redis
from fastapi import FastAPI, HTTPException, Request
from typing import Optional
import time

app = FastAPI()
redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

class RedisRateLimiter:
    \"\"\"Redis-based rate limiter using sliding window.\"\"\" 
    
    def __init__(self, redis_client):
        self.redis = redis_client
    
    def is_allowed(
        self,
        key: str,
        max_requests: int,
        window_seconds: int
    ) -> tuple[bool, dict]:
        \"\"\"Check if request is allowed.\"\"\" 
        
        now = time.time()
        window_start = now - window_seconds
        
        # Use Redis sorted set for sliding window
        pipe = self.redis.pipeline()
        
        # Remove old entries
        pipe.zremrangebyscore(key, 0, window_start)
        
        # Count requests in window
        pipe.zcard(key)
        
        # Add current request
        pipe.zadd(key, {str(now): now})
        
        # Set expiration
        pipe.expire(key, window_seconds)
        
        results = pipe.execute()
        request_count = results[1]
        
        # Check if allowed
        allowed = request_count < max_requests
        
        # Get rate limit info
        info = {
            'limit': max_requests,
            'remaining': max(0, max_requests - request_count - 1),
            'reset': int(now + window_seconds)
        }
        
        if not allowed:
            # Remove the request we just added
            self.redis.zrem(key, str(now))
        
        return allowed, info

limiter = RedisRateLimiter(redis_client)

async def rate_limit_middleware(request: Request, call_next):
    \"\"\"Rate limiting middleware.\"\"\" 
    
    # Get user identifier
    api_key = request.headers.get('X-API-Key', 'anonymous')
    rate_limit_key = f"rate_limit:{api_key}:{request.url.path}"
    
    # Check rate limit (100 requests per minute)
    allowed, info = limiter.is_allowed(rate_limit_key, max_requests=100, window_seconds=60)
    
    if not allowed:
        return JSONResponse(
            status_code=429,
            content={"error": "Rate limit exceeded"},
            headers={
                'X-RateLimit-Limit': str(info['limit']),
                'X-RateLimit-Remaining': '0',
                'X-RateLimit-Reset': str(info['reset']),
                'Retry-After': '60'
            }
        )
    
    # Add rate limit headers to response
    response = await call_next(request)
    response.headers['X-RateLimit-Limit'] = str(info['limit'])
    response.headers['X-RateLimit-Remaining'] = str(info['remaining'])
    response.headers['X-RateLimit-Reset'] = str(info['reset'])
    
    return response

app.middleware('http')(rate_limit_middleware)

@app.post("/api/chat")
async def chat(message: str):
    return {"response": "..."}

✅ Benefits of Redis:
  - Distributed rate limiting
  - Works across multiple servers
  - Persistent state
  - High performance
  - Atomic operations
""")

## 📊 Rate Limit Headers

In [ ]:
print("""
# Standard rate limit headers

from fastapi import Response
from typing import Dict

def add_rate_limit_headers(
    response: Response,
    limit: int,
    remaining: int,
    reset: int
) -> Response:
    \"\"\"Add rate limit headers to response.\"\"\" 
    
    response.headers['X-RateLimit-Limit'] = str(limit)
    response.headers['X-RateLimit-Remaining'] = str(remaining)
    response.headers['X-RateLimit-Reset'] = str(reset)
    
    return response

@app.post("/api/chat")
async def chat(message: str, response: Response):
    # Process request
    result = {"response": "..."}
    
    # Add rate limit headers
    add_rate_limit_headers(
        response,
        limit=100,
        remaining=95,
        reset=int(time.time()) + 3600
    )
    
    return result

# Example response headers:
HTTP/1.1 200 OK
X-RateLimit-Limit: 100
X-RateLimit-Remaining: 95
X-RateLimit-Reset: 1640000000
Retry-After: 60  # Only on 429 errors

# Client should:
1. Check X-RateLimit-Remaining
2. If 0, wait until X-RateLimit-Reset
3. On 429, respect Retry-After header
""")

## ✅ Summary

### Rate Limiting Algorithms:

**1. Token Bucket**
```python
# Allows bursts
bucket = TokenBucket(capacity=100, refill_rate=10)
if bucket.consume():
    process_request()

✅ Use for:
  - APIs that need burst capability
  - Gradual rate limiting
```

**2. Sliding Window**
```python
# Precise limits
limiter = SlidingWindow(max_requests=100, window=60)
if limiter.is_allowed():
    process_request()

✅ Use for:
  - Strict rate limits
  - Fair distribution
```

**3. Fixed Window**
```python
# Simple but has edge case issues
@limiter.limit("100/minute")
def endpoint():
    pass

⚠️ Can allow 2x requests at window boundary
```

### Implementation Strategies:

**Simple (In-Memory):**
```python
from slowapi import Limiter

@app.post("/api/chat")
@limiter.limit("100/hour")
async def chat(request: Request):
    return {"response": "..."}

✅ Good for:
  - Single server
  - Development
  - Simple apps
```

**Production (Redis):**
```python
limiter = RedisRateLimiter(redis_client)
allowed, info = limiter.is_allowed(key, max_requests, window)

✅ Good for:
  - Multiple servers
  - Production
  - Distributed systems
```

### Best Practices:

**1. Tiered Limits**
```python
RATE_LIMITS = {
    "free": "10/hour",
    "pro": "100/hour",
    "enterprise": "1000/hour"
}
```

**2. Different Limits per Endpoint**
```python
@app.get("/cheap")      # High limit
@limiter.limit("1000/hour")

@app.post("/expensive")  # Low limit
@limiter.limit("10/hour")
```

**3. Include Rate Limit Headers**
```python
response.headers['X-RateLimit-Limit'] = '100'
response.headers['X-RateLimit-Remaining'] = '95'
response.headers['X-RateLimit-Reset'] = '1640000000'
```

**4. Informative Error Messages**
```python
raise HTTPException(
    status_code=429,
    detail="Rate limit exceeded. Try again in 60 seconds.",
    headers={"Retry-After": "60"}
)
```

**5. Log Rate Limit Events**
```python
if not allowed:
    logger.warning(
        f"Rate limit exceeded: user={user_id} "
        f"endpoint={endpoint} limit={limit}"
    )
```

### Common Patterns:

**Per-User + Per-Endpoint:**
```python
key = f"rate_limit:{user_id}:{endpoint}"
```

**Global + Per-User:**
```python
@limiter.limit("10000/hour")  # Global
@limiter.limit("100/hour")     # Per user
```

**Cost-Based:**
```python
# Expensive operations cost more
bucket.consume(tokens=10)  # GPT-4
bucket.consume(tokens=1)   # GPT-3.5
```

### Testing:

```bash
# Test rate limiting
for i in {1..10}; do
  curl -H "X-API-Key: test" http://localhost:8000/api/chat
done

# Should see:
# Requests 1-5: 200 OK
# Request 6: 429 Too Many Requests
```

### Next: `08_production_apis/06_middleware.ipynb`